## No neural weight and no warping
NeuralBandModel vs. Sridhar et al.

In [ ]:
########## Compare neural band model to Sridhar et al. ##########

import numpy as np
import matplotlib.pyplot as plt
import decision_model as model

# First, define the location and geometry of targets in Euclidean space as a Targets object.
#    Options and necessary parameters for the geometry of the targets is given in the constructor.
#    None: delta function geometry, no other parameters needed
#    'circle': targets are circles of radius r (they can share the same radius or have different ones)
#    'capsule': targets are line segments with semicircular endcaps, length l, width w, orientation theta
# target_locs = np.array([[4.33,2.5],[4.33,-2.5]]) # (x,y) coordinates

# THREE TARGET CASE FROM PAPER
target_locs = np.array([[3.83, -3.21], [5, 0], [3.83, 3.21]])

# targets = model.Targets(locs=target_locs, geom_name=None)
targets = model.Targets(locs=target_locs, geom_name='circle', r=0.5)
# NOTE: the circular target seems to result in one extra unstable equilib in the 
#    middle of the new neural band direction mesh. This needs to be debugged.

# Define an observer location and heading in Euclidean space as a PerceptionModel object.
#    Give it knowledge of the targets (the Targets object), and define the 
#    neural band weight and angle mappings.
focal_loc = (0,0)
focal_angle = 0
percep_model = model.PerceptionModel(targets, focal_loc, focal_angle, 
                                     neural_angle_dist=None, angle_weight=None)

# Plot the perception model to see the current geometry and perception signal
percep_model.plot_blocked_signals(wb_plot=True)

In [ ]:
### Plot the direction mesh bifurcation diagram and compare to Sridhar ###

# Create the neural band model, which will use the above information to compute 
#     consensus directions.
# beta is the neural Boltzmann factor; the earlier parameterization used a
# temperature T=0.2 whose effective coupling was N_targets/T, so the 3-target
# case active above needs beta = 3/0.2 = 15 (the default 10 is the 2-target value).

# Compare the new model to the Sridhar model.
dir_model_Sridhar = model.NeuralBandModel(percep_model, beta=target_locs.shape[0]/0.2, angle_distortion_nu=0.5)
# Note: the below neur_model is *unwarped*. To recover biological behavior, 
#   angle warping is necessary.
neur_model = model.NeuralBandModel(percep_model, beta=target_locs.shape[0]/0.2)

plt.figure(figsize=(12,6))
ax1 = plt.subplot(121)
ax2 = plt.subplot(122)

from multiprocessing import Pool
with Pool(10) as pool:
    dir_model_Sridhar.plot_direction_mesh(pool=pool, wb_plot=True, ax=ax1)
    neur_model.plot_direction_mesh(pool=pool, wb_plot=True, ax=ax2)

plt.show()

## Cutoff warping only

In [ ]:
####### Test the warped neural band against Sridhar nu model #######

import numpy as np
import matplotlib.pyplot as plt
import decision_model as model

target_locs = np.array([[4.33,2.5],[4.33,-2.5]]) # (x,y) coordinates

# targets = model.Targets(locs=target_locs, geom_name=None)
targets = model.Targets(locs=target_locs, geom_name='circle', r=0.5)

focal_loc = (0,0)
focal_angle = 0
percep_model = model.PerceptionModel(targets, focal_loc, focal_angle, 
                                     neural_angle_dist=None, angle_weight=None)
percep_model_warped = model.PerceptionModel(targets, focal_loc, focal_angle,
                                            neural_angle_dist='lin_cutoff', 
                                            angle_weight=None, a_warp=np.pi/4,
                                            b_warp=np.pi)

# Plot the perception model to see the current geometry and perception signal
percep_model_warped.plot_blocked_signals(wb_plot=True)

In [ ]:
dir_model = model.NeuralBandModel(percep_model, angle_distortion_nu=0.5)
neur_model = model.NeuralBandModel(percep_model_warped)

plt.figure(figsize=(12,6))
ax1 = plt.subplot(121)
ax2 = plt.subplot(122)

from multiprocessing import Pool
with Pool(10) as pool:
    dir_model.plot_direction_mesh(pool=pool, wb_plot=True, ax=ax1, title='Sridhar et al.')
    neur_model.plot_direction_mesh(pool=pool, wb_plot=True, ax=ax2)

plt.show()

## Cutoff weight only

In [ ]:
####### Test the effect of weighting the neural band (no warping) #######

import numpy as np
import matplotlib.pyplot as plt
import decision_model as model

target_locs = np.array([[4.33,2.5],[4.33,-2.5]]) # (x,y) coordinates

targets = model.Targets(locs=target_locs, geom_name=None)
# targets = model.Targets(locs=target_locs, geom_name='circle', r=0.5)

focal_loc = (0,0)
focal_angle = 0
percep_model = model.PerceptionModel(targets, focal_loc, focal_angle, 
                                     neural_angle_dist=None, angle_weight=None)
percep_model_weighted = model.PerceptionModel(targets, focal_loc, focal_angle,
                                        neural_angle_dist=None, 
                                        angle_weight='lin_cutoff', 
                                        a_weight=np.pi/4, b_weight=np.pi)

# Plot the perception model to see the current geometry and perception signal
percep_model_weighted.plot_blocked_signals(wb_plot=True)

In [ ]:
base_model = model.NeuralBandModel(percep_model)
neur_model = model.NeuralBandModel(percep_model_weighted)

plt.figure(figsize=(12,6))
ax1 = plt.subplot(121)
ax2 = plt.subplot(122)

from multiprocessing import Pool
with Pool(10) as pool:
    base_model.plot_direction_mesh(pool=pool, wb_plot=True, ax=ax1, title='Base model')
    neur_model.plot_direction_mesh(pool=pool, wb_plot=True, ax=ax2, title='Weighted model')

plt.show()

## Cutoff weight and integral warping

In [ ]:
####### Compare the weighted and warped neural band with Sridhar nu model #######

import numpy as np
import matplotlib.pyplot as plt
import decision_model as model

target_locs = np.array([[4.33,2.5],[4.33,-2.5]]) # (x,y) coordinates

# targets = model.Targets(locs=target_locs, geom_name=None)
targets = model.Targets(locs=target_locs, geom_name='circle', r=0.5)

focal_loc = (0,0)
focal_angle = 0
percep_model = model.PerceptionModel(targets, focal_loc, focal_angle, 
                                     neural_angle_dist=None, angle_weight=None)
percep_model_both = model.PerceptionModel(targets, focal_loc, focal_angle,
                                          neural_angle_dist='lin_cutoff', 
                                          angle_weight='neural_angle_dist', # same as warp
                                          a_warp=np.pi/4, b_warp=np.pi)

# Plot the perception model to see the current geometry and perception signal
percep_model_both.plot_blocked_signals(wb_plot=True)

In [ ]:
dir_model = model.NeuralBandModel(percep_model, angle_distortion_nu=0.5)
neur_model = model.NeuralBandModel(percep_model_both)

plt.figure(figsize=(12,6))
ax1 = plt.subplot(121)
ax2 = plt.subplot(122)

from multiprocessing import Pool
with Pool(10) as pool:
    dir_model.plot_direction_mesh(pool=pool, wb_plot=True, ax=ax1, title='Sridhar model')
    neur_model.plot_direction_mesh(pool=pool, wb_plot=True, ax=ax2, title='Weighted and warped model')

plt.show()